# Distribution Shape vs Redshift — Testing for Physical Evolution

**Question:** Do the shape parameters of the u−r colour distribution change with redshift
more than expected from sampling noise alone?

**Why it matters:** Any purely random process produces a Normal distribution by the CLT.
If the shape of the colour distribution (skewness, kurtosis — captured by Johnson SU
parameters a and b, or Box-Cox λ) evolves with redshift *beyond what sampling variation
predicts*, that is evidence of a physical process at work — not just noise.

**Method:**
1. Fit Johnson SU and Box-Cox to u−r in redshift bins.
2. Bootstrap the null: draw same-sized subsamples from the *full* sample, refit,
   build the distribution of shape parameters expected under 'no evolution'.
3. Plot observed shape parameters vs z with the null 95% confidence envelope.
4. If observed values fall outside the envelope → variation is not just noise.

**This directly answers the statistician's objection** that differences are 'typical
statistical variation' — the bootstrap quantifies exactly what typical looks like.


In [ ]:
from astropy.table import Table
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

## 1. Load Data

In [ ]:
t = Table.read('../../GAMA_DATA/StellarMassesLambdarv24.fits')

# Quality cuts
t = t[t['uminusr'] > 0.5]   # remove unphysical blue outliers
t = t[t['uminusr'] < 4.0]   # remove extreme red outliers / bad photometry
t = t[t['logmstar'] > 8.0]  # remove very low mass (poorly constrained SED fits)
t = t[t['Z'] > 0.002]       # remove local volume / peculiar velocity dominated
t = t[t['Z'] < 0.35]        # GAMA main survey limit

uminusr = np.array(t['uminusr'], dtype=float)
redshift = np.array(t['Z'], dtype=float)

print(f'Total sample after cuts: {len(uminusr):,}')
print(f'Redshift range: {redshift.min():.3f} – {redshift.max():.3f}')
print(f'u−r range:      {uminusr.min():.3f} – {uminusr.max():.3f}')

## 2. Define Redshift Bins

In [ ]:
# Equal-width bins — adjust if any bin has N < 300
z_edges = np.array([0.002, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35])
z_centres = 0.5 * (z_edges[:-1] + z_edges[1:])
n_bins = len(z_centres)

bin_masks = [(redshift >= z_edges[i]) & (redshift < z_edges[i+1])
             for i in range(n_bins)]
bin_sizes = [mask.sum() for mask in bin_masks]

print(f'{"z bin":>18}  {"N":>7}')
print('-' * 30)
for i, (zlo, zhi, n) in enumerate(zip(z_edges[:-1], z_edges[1:], bin_sizes)):
    flag = '  *** small' if n < 300 else ''
    print(f'  {zlo:.3f} – {zhi:.3f}     {n:>7,}{flag}')

## 3. Fit Functions

In [ ]:
def fit_johnsonsu(x):
    """Fit Johnson SU distribution. Returns (a, b, loc, scale)."""
    try:
        a, b, loc, scale = stats.johnsonsu.fit(x)
        return a, b, loc, scale
    except Exception:
        return np.nan, np.nan, np.nan, np.nan

def fit_boxcox(x):
    """Fit Box-Cox lambda (best normalising power transform). Returns lambda."""
    try:
        # Box-Cox requires positive data; shift if needed
        shift = max(0, -x.min() + 0.01)
        _, lam = stats.boxcox(x + shift)
        return lam
    except Exception:
        return np.nan

def fit_skew_kurt(x):
    """Return sample skewness and excess kurtosis."""
    return stats.skew(x), stats.kurtosis(x)

print('Fit functions defined')

## 4. Fit Distributions in Each Redshift Bin

In [ ]:
obs_a   = []   # Johnson SU shape param a (skewness-related)
obs_b   = []   # Johnson SU shape param b (kurtosis-related)
obs_lam = []   # Box-Cox lambda
obs_skew= []
obs_kurt= []
valid_bins = []

print(f'{"z centre":>10} {"N":>7} {"JSU a":>8} {"JSU b":>8} {"BoxCox λ":>10} {"Skew":>8} {"Kurt":>8}')
print('-' * 65)

for i, (mask, zc) in enumerate(zip(bin_masks, z_centres)):
    x = uminusr[mask]
    n = len(x)
    if n < 200:
        print(f'{zc:>10.3f} {n:>7,}   (skipped — too few)')
        continue
    a, b, _, _ = fit_johnsonsu(x)
    lam        = fit_boxcox(x)
    skw, krt   = fit_skew_kurt(x)
    obs_a.append(a); obs_b.append(b)
    obs_lam.append(lam)
    obs_skew.append(skw); obs_kurt.append(krt)
    valid_bins.append(zc)
    print(f'{zc:>10.3f} {n:>7,} {a:>8.3f} {b:>8.3f} {lam:>10.3f} {skw:>8.3f} {krt:>8.3f}')

valid_bins = np.array(valid_bins)
obs_a    = np.array(obs_a)
obs_b    = np.array(obs_b)
obs_lam  = np.array(obs_lam)
obs_skew = np.array(obs_skew)
obs_kurt = np.array(obs_kurt)

## 5. Bootstrap the Null — What Does Sampling Variation Look Like?

In [ ]:
N_BOOT = 500  # publication-quality

null_a    = {zc: [] for zc in valid_bins}
null_b    = {zc: [] for zc in valid_bins}
null_lam  = {zc: [] for zc in valid_bins}
null_skew = {zc: [] for zc in valid_bins}
null_kurt = {zc: [] for zc in valid_bins}

# Bin sizes for valid bins
valid_ns = [bin_masks[i].sum() 
            for i, zc in enumerate(z_centres) 
            if zc in valid_bins and bin_masks[i].sum() >= 200]

rng = np.random.default_rng(42)

print(f'Bootstrapping null distribution ({N_BOOT} iterations per bin)...')
for boot_i in range(N_BOOT):
    for zc, n in zip(valid_bins, valid_ns):
        # Draw random subsample of same size from full dataset
        idx = rng.choice(len(uminusr), size=n, replace=True)
        x   = uminusr[idx]
        a, b, _, _ = fit_johnsonsu(x)
        lam         = fit_boxcox(x)
        skw, krt    = fit_skew_kurt(x)
        null_a[zc].append(a)
        null_b[zc].append(b)
        null_lam[zc].append(lam)
        null_skew[zc].append(skw)
        null_kurt[zc].append(krt)
    if (boot_i + 1) % 100 == 0:
        print(f'  {boot_i+1}/{N_BOOT} done')

print('Bootstrap complete.')

## 6. Plot: Observed Shape Parameters vs Redshift with Null Envelope

In [ ]:
def plot_with_null(ax, zc, obs, null_dict, ylabel, title, color='steelblue'):
    lo = np.array([np.percentile(null_dict[z], 2.5)  for z in zc])
    hi = np.array([np.percentile(null_dict[z], 97.5) for z in zc])
    med= np.array([np.percentile(null_dict[z], 50)   for z in zc])

    ax.fill_between(zc, lo, hi, alpha=0.25, color='grey', label='Null 95% CI')
    ax.plot(zc, med, '--', color='grey', lw=1, label='Null median')
    ax.plot(zc, obs, 'o-', color=color, lw=2, ms=7, label='Observed')

    # Mark bins where observed falls outside null envelope
    outside = (obs < lo) | (obs > hi)
    if outside.any():
        ax.plot(zc[outside], obs[outside], '*', color='red', ms=12,
                zorder=5, label='Outside null 95% CI')

    ax.set_xlabel('Redshift z', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=150)
fig.suptitle('u−r Colour Distribution Shape vs Redshift\n'
             'Grey band = 95% null envelope (bootstrap from full sample)\n'
             'Red stars = bins where observed shape is inconsistent with sampling noise',
             fontsize=11)

plot_with_null(axes[0,0], valid_bins, obs_a,    null_a,    'Johnson SU  a',  'JSU shape param a (skewness)',    'steelblue')
plot_with_null(axes[0,1], valid_bins, obs_b,    null_b,    'Johnson SU  b',  'JSU shape param b (kurtosis)',    'steelblue')
plot_with_null(axes[0,2], valid_bins, obs_lam,  null_lam,  'Box-Cox  λ',     'Box-Cox lambda (normalising power)','darkorange')
plot_with_null(axes[1,0], valid_bins, obs_skew, null_skew, 'Skewness',       'Sample skewness',                 'seagreen')
plot_with_null(axes[1,1], valid_bins, obs_kurt, null_kurt, 'Excess kurtosis','Sample excess kurtosis',          'seagreen')

# Summary panel: fraction of parameters outside null per bin
ax = axes[1,2]
all_obs   = [obs_a, obs_b, obs_lam, obs_skew, obs_kurt]
all_nulls = [null_a, null_b, null_lam, null_skew, null_kurt]
frac_outside = []
for zc in valid_bins:
    n_out = 0
    for obs_arr, null_dict in zip(all_obs, all_nulls):
        idx = list(valid_bins).index(zc)
        lo = np.percentile(null_dict[zc], 2.5)
        hi = np.percentile(null_dict[zc], 97.5)
        if obs_arr[idx] < lo or obs_arr[idx] > hi:
            n_out += 1
    frac_outside.append(n_out / 5)

colors = ['red' if f > 0 else 'steelblue' for f in frac_outside]
ax.bar(valid_bins, frac_outside, width=0.04, color=colors, alpha=0.7)
ax.axhline(0.05, color='grey', linestyle='--', label='5% expected by chance')
ax.set_xlabel('Redshift z', fontsize=10)
ax.set_ylabel('Fraction of shape params\noutside null 95% CI', fontsize=10)
ax.set_title('Summary: signal strength per z-bin', fontsize=10)
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../ChartsPlots/DistributionShapeVsRedshift.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 7. Formal Test: Are Shape Parameters Monotonically Trending with z?

In [ ]:
from scipy.stats import spearmanr, kendalltau

print('Spearman correlation of shape parameters with redshift')
print('(Tests for monotonic trend — physical evolution would produce one)\n')
print(f'{"Parameter":<20} {"Spearman r":>12} {"p-value":>12} {"Kendall tau":>12} {"p-value":>12}')
print('-' * 72)

params = [
    ('Johnson SU  a',    obs_a),
    ('Johnson SU  b',    obs_b),
    ('Box-Cox lambda',   obs_lam),
    ('Skewness',         obs_skew),
    ('Excess kurtosis',  obs_kurt),
]
for name, obs in params:
    sr, sp = spearmanr(valid_bins, obs)
    kt, kp = kendalltau(valid_bins, obs)
    sig = ' *' if sp < 0.05 else ''
    print(f'{name:<20} {sr:>12.4f} {sp:>12.4e} {kt:>12.4f} {kp:>12.4e}{sig}')

print()
print('* p < 0.05: monotonic trend detected')
print()
print('Interpretation:')
print('  Significant trend + observed values outside null envelope')
print('  → shape evolution is real, not sampling noise')
print('  No significant trend + values inside null envelope')
print('  → statistician was right; no detectable physical signal in shape')

## 8. Physical Interpretation of Box-Cox Lambda

In [ ]:
print('Box-Cox lambda interpretation guide:')
print()
print('  λ ≈  1.0  →  data already approximately Normal (additive process)')
print('  λ ≈  0.0  →  log-Normal (multiplicative/exponential process)')
print('  λ ≈ -1.0  →  reciprocal transform best; strongly right-bounded distribution')
print('  λ <  0    →  long left tail suppressed; consistent with one-way quenching ratchet')
print()
print('  For u−r colour:')
print('  λ trending toward 0 at higher z → distribution becoming more log-Normal')
print('  (more galaxies in exponential decline / transition at earlier times)')
print('  λ trending negative at lower z  → distribution more sharply bounded')
print('  (red sequence consolidating; fewer transition objects)')
print()
for zc, lam in zip(valid_bins, obs_lam):
    print(f'  z = {zc:.3f}  λ = {lam:.3f}')

## 9. Red Fraction vs Box-Cox λ

Red fraction = fraction of galaxies with u−r > 2.1 in each redshift bin.
Tests whether λ tracks the redness of the population directly.
λ = 1 is the Normal point; expect red fraction ≈ 0.5 near that bin.

In [ ]:
# Red fraction per redshift bin (only valid bins used in shape analysis)
RED_THRESHOLD = 2.1  # standard u-r red/blue dividing line

red_fractions = []
for zc, mask in zip(z_centres, bin_masks):
    if zc not in valid_bins:
        continue
    colours = t['uminusr'][mask]
    rf = float(np.sum(colours > RED_THRESHOLD)) / len(colours)
    red_fractions.append(rf)

red_fractions = np.array(red_fractions)
lam_arr       = np.array(obs_lam)
z_arr         = np.array(valid_bins)

print(f"{'z':>8}  {'N':>7}  {'Red frac':>10}  {'lambda':>8}")
print('-' * 42)
for zc, mask, rf, lam in zip(z_centres, bin_masks, red_fractions, lam_arr):
    if zc not in valid_bins:
        continue
    print(f"{zc:8.3f}  {mask.sum():7d}  {rf:10.3f}  {lam:8.3f}")

# Spearman correlation
from scipy.stats import spearmanr
r, p = spearmanr(red_fractions, lam_arr)
print(f'
Spearman r(red fraction, lambda) = {r:.4f},  p = {p:.3e}')

# --- Scatter plot ---
fig, ax = plt.subplots(figsize=(6, 5))

sc = ax.scatter(red_fractions, lam_arr, c=z_arr, cmap='viridis',
                s=90, zorder=5, edgecolors='k', linewidths=0.5)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Redshift z', fontsize=11)

for rf, lam, zc in zip(red_fractions, lam_arr, z_arr):
    ax.annotate(f'z={zc:.2f}', (rf, lam),
                textcoords='offset points', xytext=(6, 2), fontsize=8)

# Reference lines
ax.axhline(1.0, color='grey', linewidth=0.8, linestyle='--', alpha=0.7,
           label=r'$\lambda$ = 1  (Normal)')
ax.axvline(0.5, color='salmon', linewidth=0.8, linestyle='--', alpha=0.7,
           label='Red fraction = 0.5')

ax.set_xlabel('Red Fraction  (u−r > 2.1)', fontsize=12)
ax.set_ylabel(r'Box-Cox $\lambda$', fontsize=12)
ax.set_title(
    f'Red Fraction vs Box-Cox λ  (GAMA DR4)
'
    f'Spearman r = {r:.3f},  p = {p:.2e}',
    fontsize=11
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../ChartsPlots/RedFraction_vs_Lambda.png', dpi=150,
            bbox_inches='tight')
plt.show()
print('Saved: ChartsPlots/RedFraction_vs_Lambda.png')
